# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the TensorDict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` holds preset constructors (`cartpole`, `procedural_frozenlake`, `synthetic`, `atari`, …). `make_vector_env` wraps Gymnasium and formats steps as `(data, metadata, metrics)`.

In [5]:
from tensordict import TensorDict as TD
from mouse_envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=1` runs four CartPole instances in parallel.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [6]:
cfg = EnvConfig.cartpole(seed=0, num_envs=1, max_episode_steps=500)
env = make_vector_env(cfg)

## Rollout loop

Each `env.step` returns:

- **`data[i]`** — `time`, `observation` (dict: `discrete` / `continuous` / `image`), `reward`, `done`, …
- **`metadata[i]`** — per-env training context: `group_id`, `episode_index`, `reward_episodic`, optional `q_star`
- **`metrics[i]`** — `episode_cum_reward` and `episode_length` lists for finishes on this step (empty if none)

Each **`actions[i]["action"]`** passed to `step()` is also a dict — `discrete` or `continuous`, matching the env's action space.

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [7]:
for step in range(1000):
    actions = env.sample_random_actions()
    data, metadata, metrics = env.step(actions)

    for i, td in enumerate(data):
        meta = metadata[i]
        obs = {k: v.tolist() for k, v in TD(td["observation"]).items()}
        act = {k: v.tolist() for k, v in TD(actions[i]["action"]).items()}
        print(
            f"step={step:4d}  "
            f"group_id={meta['group_id']}  "
            f"time={td['time'].item()}  "
            f"act={act}  "
            f"done={td['done'].item()}  "
            f"reward={td['reward'].item():.3f}  "
            f"reward_episodic={meta['reward_episodic']:.3f}  "
            f"obs={obs}"
        )

step=   0  group_id=CartPole-v1  time=0  act={'discrete': [1]}  done=0  reward=0.000  reward_episodic=0.000  obs={'continuous': [-0.014679748564958572, 0.01799100451171398, 0.0375664159655571, -0.02614838071167469]}
step=   1  group_id=CartPole-v1  time=1  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.002  obs={'continuous': [-0.014319928362965584, 0.21255464851856232, 0.03704344481229782, -0.3067460060119629]}
step=   2  group_id=CartPole-v1  time=2  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.006  obs={'continuous': [-0.010068835690617561, 0.40712970495224, 0.030908526852726936, -0.5875200629234314]}
step=   3  group_id=CartPole-v1  time=3  act={'discrete': [0]}  done=0  reward=1.000  reward_episodic=0.010  obs={'continuous': [-0.0019262413261458278, 0.21158882975578308, 0.019158124923706055, -0.28526321053504944]}
step=   4  group_id=CartPole-v1  time=4  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.014  obs={'continuous': [0.0023

## Cleanup

In [8]:
env.close()